In [ ]:
from datetime import datetime

STARTED_AT = datetime.now()
print(f"Started: {STARTED_AT}")

BRONZE_VOLUME_PATH = (
    "/Volumes/dbw_stratum_dev/bronze/"
    "nyc_taxi/yellow/2024/01/"
)

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "dbw_stratum_dev.bronze"
)
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "dbw_stratum_dev.silver"
)
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "dbw_stratum_dev.gold"
)

print("Schemas verified")

print(f"\nReading Delta files from volume...")

df = spark.read.format("delta").load(
    BRONZE_VOLUME_PATH
)

rows_read = df.count()
print(f"Rows read: {rows_read:,}")
print("\nWriting as managed Unity Catalog table...")

(df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "dbw_stratum_dev.bronze.nyc_taxi_yellow"
    ))

print("Table created: "
      "dbw_stratum_dev.bronze.nyc_taxi_yellow")

row_count = spark.sql(
    "SELECT COUNT(*) as row_count "
    "FROM dbw_stratum_dev.bronze.nyc_taxi_yellow"
).collect()[0]["row_count"]

print(f"Row count: {row_count:,}")

assert row_count == 2_964_624, (
    f"Expected 2,964,624 rows but got {row_count:,}."
)

print("Row count assertion passed")

display(spark.sql("""
    SELECT
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        fare_amount,
        total_amount
    FROM dbw_stratum_dev.bronze.nyc_taxi_yellow
    LIMIT 10
"""))

completed_at = datetime.now()
duration = (completed_at - STARTED_AT).total_seconds()

print(f"\nRegistration complete")
print(f"Duration: {duration:.1f}s")
print(f"Rows: {row_count:,}")
print(f"\ndbt sources can now reference:")
print(f"  dbw_stratum_dev.bronze.nyc_taxi_yellow")